In [1]:
import torch.nn as nn
import torch
import random
import math
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as transforms
import os
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

dataset = "/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset"
image_dir = os.path.join(dataset, "images", "Images")

transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image_paths = []
image_labels = []

for i, e in enumerate(sorted(os.listdir(image_dir))):
    breed_dir = os.path.join(image_dir, e)
    for f in os.listdir(breed_dir):
        image_paths.append(os.path.join(breed_dir, f))
        image_labels.append(i)

print(len(image_paths), len(image_labels))

image_paths_training, image_paths_test, labels_train, labels_test = train_test_split(
    image_paths, image_labels, test_size=0.2, random_state=42, stratify=image_labels
)

class DogDataset(Dataset):
    def __init__(self, image_paths, labels, transform):
        super().__init__()
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __getitem__(self, idex):
        path = self.image_paths[idex]
        img = Image.open(path).convert("RGB")
        imag = self.transform(img)
        label = self.labels[idex]
        return imag, label

    def __len__(self):
        return len(self.image_paths)

train_objects = DogDataset(image_paths_training, labels_train, transform_pipeline)
test_objects = DogDataset(image_paths_test, labels_test, transform_pipeline)

train_loader = DataLoader(train_objects, batch_size=64, shuffle=True)
test_loader = DataLoader(test_objects, batch_size=64, shuffle=False)

train = iter(train_loader)
train_images, train_labels = next(train)
print(train_images.shape)
print(train_labels.shape)

test = iter(test_loader)
test_images, test_labels = next(test)
print(test_images.shape)
print(test_labels.shape)
class DogClassifier(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,kernel_size=3,padding=1)
        self.maxpol1=nn.MaxPool2d(kernel_size=2)
        self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
        self.maxpol2=nn.MaxPool2d(kernel_size=2)
        self.conv3=nn.Conv2d(64,128,kernel_size=3,padding=1)
        self.relu=nn.ReLU()
        self.maxpol3=nn.MaxPool2d(kernel_size=2)
        self.fc=nn.Linear(128*28*28,num_classes)
    def forward(self, x):
        x = self.maxpol1(self.relu(self.conv1(x)))
        x = self.maxpol2(self.relu(self.conv2(x)))
        x = self.maxpol3(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
device="cuda" if torch.cuda.is_available() else "cpu"
model=DogClassifier(num_classes=120)
model=model.to(device)
print(device)
optimizer=torch.optim.Adam((model.parameters()),lr=0.0001)
criterion=nn.CrossEntropyLoss()
num_epochs=10
for epoch in range(num_epochs):
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        outputs=model(images)
        optimizer.zero_grad()
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        print((outputs.argmax(dim=1) == labels).float().mean().item())
        
        
        

20580 20580
torch.Size([64, 3, 224, 224])
torch.Size([64])
torch.Size([64, 3, 224, 224])
torch.Size([64])
cuda
0.0
0.0
0.015625
0.0
0.0
0.015625
0.015625
0.0
0.0
0.0
0.0
0.015625
0.0
0.03125
0.015625
0.015625
0.0
0.015625
0.015625
0.0
0.0
0.0
0.015625
0.015625
0.015625
0.0625
0.0
0.015625
0.0
0.0
0.03125
0.015625
0.0
0.0
0.03125
0.0
0.0
0.015625
0.03125
0.015625
0.03125
0.015625
0.0
0.0
0.03125
0.03125
0.03125
0.015625
0.046875
0.0625
0.03125
0.046875
0.015625
0.015625
0.0
0.015625
0.0
0.015625
0.015625
0.03125
0.015625
0.046875
0.046875
0.015625
0.03125
0.0
0.03125
0.015625
0.015625
0.0625
0.046875
0.0
0.0
0.015625
0.046875
0.015625
0.0625
0.015625
0.015625
0.046875
0.0
0.0625
0.046875
0.015625
0.03125
0.03125
0.046875
0.03125
0.109375
0.03125
0.046875
0.03125
0.046875
0.078125
0.046875
0.0625
0.03125
0.0
0.03125
0.046875
0.046875
0.0
0.015625
0.03125
0.03125
0.03125
0.03125
0.09375
0.09375
0.015625
0.015625
0.03125
0.109375
0.015625
0.015625
0.03125
0.03125
0.03125
0.015625
0.015625
